In [ ]:
# Install packages if needed
# Uncomment and run this cell in a fresh environment.

# !pip install tensorflow opencv-python matplotlib numpy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("OpenCV version:", cv2.__version__)

In [ ]:
VIDEO_PATH = "sample_video.mp4"

# Open the video
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("Could not open the video.")
    print("Check that VIDEO_PATH is correct.")
else:
    print("Video opened successfully.")

cap.release()

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = frame_count / fps if fps > 0 else 0

print(f"FPS: {fps:.2f}")
print(f"Frame count: {frame_count}")
print(f"Resolution: {width} x {height}")
print(f"Approximate duration: {duration:.2f} seconds")

cap.release()

## 5. Read the video frame-by-frame

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

frames_read = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frames_read += 1

cap.release()

print("Frames successfully read:", frames_read)

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

ret, frame = cap.read()

cap.release()

if ret:
    # OpenCV stores images as BGR.
    # Matplotlib expects RGB.
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 6))
    plt.imshow(frame_rgb)
    plt.axis("off")
    plt.title("First Video Frame")
    plt.show()
else:
    print("Could not read the first frame.")

In [ ]:
def extract_frames(video_path, num_frames=12):
    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return []

    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()

        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

    cap.release()

    return frames


frames = extract_frames(VIDEO_PATH, num_frames=12)

print("Number of extracted frames:", len(frames))

In [ ]:
def display_frames(frames, cols=4):
    rows = int(np.ceil(len(frames) / cols))

    plt.figure(figsize=(14, 3.5 * rows))

    for i, frame in enumerate(frames):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(frame)
        plt.title(f"Frame {i + 1}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


display_frames(frames)

In [ ]:
# Resize extracted frames
IMG_SIZE = (224, 224)

processed_frames = []

for frame in frames:
    resized = cv2.resize(frame, IMG_SIZE)
    processed_frames.append(resized)

video_tensor = tf.convert_to_tensor(processed_frames, dtype=tf.float32)

print("Tensor shape:", video_tensor.shape)
print("Tensor dtype:", video_tensor.dtype)

In [ ]:
video_tensor_normalized = video_tensor / 255.0

print("Minimum:", tf.reduce_min(video_tensor_normalized).numpy())
print("Maximum:", tf.reduce_max(video_tensor_normalized).numpy())

In [ ]:
NUM_FRAMES = 16
IMG_SIZE = (112, 112)


def load_video(video_path, num_frames=NUM_FRAMES, img_size=IMG_SIZE):
    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        raise ValueError("Could not read video or video contains no frames.")

    indices = np.linspace(
        0, total_frames - 1, num_frames
    ).astype(int)

    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()

        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, img_size)
            frame = frame.astype(np.float32) / 255.0
            frames.append(frame)
        else:
            # If a frame cannot be read, use a black frame.
            frames.append(np.zeros((*img_size, 3), dtype=np.float32))

    cap.release()

    return np.array(frames, dtype=np.float32)


video = load_video(VIDEO_PATH)

print("Video shape:", video.shape)

In [ ]:
sample_indices = np.linspace(0, NUM_FRAMES - 1, 8).astype(int)

plt.figure(figsize=(14, 7))

for i, idx in enumerate(sample_indices):
    plt.subplot(2, 4, i + 1)
    plt.imshow(video[idx])
    plt.title(f"Frame {idx}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(NUM_FRAMES, IMG_SIZE[0], IMG_SIZE[1], 3)
    ),

    tf.keras.layers.Conv3D(
        filters=32,
        kernel_size=(3, 3, 3),
        activation="relu"
    ),
    tf.keras.layers.MaxPool3D(
        pool_size=(1, 2, 2)
    ),

    tf.keras.layers.Conv3D(
        filters=64,
        kernel_size=(3, 3, 3),
        activation="relu"
    ),
    tf.keras.layers.MaxPool3D(
        pool_size=(2, 2, 2)
    ),

    tf.keras.layers.Conv3D(
        filters=128,
        kernel_size=(3, 3, 3),
        activation="relu"
    ),

    tf.keras.layers.GlobalAveragePooling3D(),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    # Example: 3 classes
    tf.keras.layers.Dense(3, activation="softmax")
])

model.summary()

In [ ]:
# Example dataset creation

X = np.expand_dims(video, axis=0)

# Example label: class 0
y = np.array([0])

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((X, y))

dataset = (
    dataset
    .shuffle(10)
    .batch(1)
    .prefetch(tf.data.AUTOTUNE)
)

for batch_x, batch_y in dataset:
    print("Batch video shape:", batch_x.shape)
    print("Batch labels:", batch_y.numpy())

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

In [ ]:
# Run this locally if your environment supports OpenCV GUI windows.

# cap = cv2.VideoCapture(VIDEO_PATH)

# while True:
#     ret, frame = cap.read()
#
#     if not ret:
#         break
#
#     cv2.imshow("Video", frame)
#
#     # Press q to quit
#     if cv2.waitKey(25) & 0xFF == ord("q"):
#         break
#
# cap.release()
# cv2.destroyAllWindows()

In [ ]:
# Basic webcam example
# Run this locally.

# cap = cv2.VideoCapture(0)

# while True:
#     ret, frame = cap.read()
#
#     if not ret:
#         print("Could not read webcam.")
#         break
#
#     cv2.imshow("Webcam", frame)
#
#     # Press q to quit
#     if cv2.waitKey(1) & 0xFF == ord("q"):
#         break
#
# cap.release()
# cv2.destroyAllWindows()